In [ ]:
import torch
import torch.nn as nn
import os
from torch.utils.tensorboard import SummaryWriter 
from main_model import EndToEndDrivingPipeline, FrontViewSwinTiny
from CrosAttention import TemporalCrossAttention
from torch.utils.data import DataLoader, random_split
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from dataset_create import prepare_multiple_datasets_and_scaler, ProcessedDrivingDataset, inverse_transform

In [ ]:
class AutomaticWeightedLoss(nn.Module):
    """
    基于同方差不确定性的多任务动态 Loss 自适应权重权重分配
    """
    def __init__(self, num_tasks=2):
        super(AutomaticWeightedLoss, self).__init__()
        # 针对每个任务定义一个可学习的参数 log(sigma^2)，初始为 0
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, loss_velocity, loss_steer):
        log_var_vel = self.log_vars[0]
        log_var_steer = self.log_vars[1]

        precision_vel = torch.exp(-log_var_vel)
        precision_steer = torch.exp(-log_var_steer)

        # 动态加权 Loss：精度 * 误差 + 正则化项
        loss = (precision_vel * loss_velocity + log_var_vel) + \
               (precision_steer * loss_steer + log_var_steer)
        return loss

## 训练设定

In [ ]:


def train(input_files, output_csv = 'processed_data.csv',
            scaler_json_path = 'scaler_params.json', 
            training = True, resume_path=None):

    target_names = ['velocity', 'steer']    

    # 数据预处理
    print(">>> 开始检查并预处理数据...")
    val_output_csv = 'val_data_csv'
    val_scaler_json_path = 'val_scaler_json'
    scaler_params = prepare_multiple_datasets_and_scaler(
        input_files=input_files,
        output_csv=output_csv,
        scaler_json_path=scaler_json_path,
        seq_length=9)
    print(">>> 预处理完成！\n")

    # 加载数据集
    full_dataset = ProcessedDrivingDataset(csv_file=output_csv)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    # val_dataset = ProcessedDrivingDataset(csv_file=val_output_csv)
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

    # 初始化模型、损失函数、优化器
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    unet_weight_path = r'dpai_unet.pth'

    model = EndToEndDrivingPipeline(unet_weight_path=unet_weight_path, device=device, output_dim=2)# 因你的预测只有速度[0]和转角[1]两个数，设为2这极为关键！
    model.to(device)

    awl = AutomaticWeightedLoss(num_tasks=2).to(device)

    optimizer = Adam([
        {'params': model.parameters(), 'lr': 1e-4},
        {'params': awl.parameters(), 'lr': 1e-3}
    ])
    scheduler = StepLR(optimizer, step_size=10, gamma=0.9)
    criterion_huber = nn.HuberLoss(delta=0.1)

    # 设置加权损失的权重超参数
    Loss_name = 'HuberLoss'
    start_epoch = 0
    if resume_pth and os.path.exists(resume_pth):
        print(f">>> 正在读取断点文件: {resume_pth}")
        checkpoint = torch.load(resume_pth, map_location=device)
        
        # 加载权重
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if 'awl_state_dict' in checkpoint:
            awl.load_state_dict(checkpoint['awl_state_dict'])
        
        # 记录上次训练在那一个 Epoch
        start_epoch = checkpoint['epoch']
        print(f">>> 已成功恢复训练！将从第 {start_epoch + 1} 个 Epoch 开始继续...")

    # 日志文件
    log_dir = "training_results1"
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f"{Loss_name}.txt")
    writer = SummaryWriter(os.path.join(log_dir, "SwinTiny_logs"))

    # 训练循环
    num_epochs = 50
    global_step = start_epoch * len(train_loader)
    if training:
        for epoch in range(num_epochs):
            model.train()
            train_sum_loss, train_sum_vel, train_sum_steer = 0.0, 0.0, 0.0

            for batch_idx, (front_imgs, side_imgs, state_seq, target) in enumerate(train_loader):
                # 数据解包并推上显存
                front_imgs = [img.to(device) for img in front_imgs]
                side_imgs = [side_img.to(device) for side_img in side_imgs]
                state_seq = state_seq.to(device)
                target = target.to(device)
                # �� 第三重修改：前向传播！一针入魂：(3张纯前向RGB组, 1张纯左侧RGB, 8帧10维历史流)
                optimizer.zero_grad()
                predictions = model(front_imgs, side_imgs, state_seq)

                batch_v_loss = criterion_huber(predictions[:, 0], target[:, 0])
                batch_s_loss = criterion_huber(predictions[:, 1], target[:, 1])
                total_loss = awl(batch_v_loss, batch_s_loss)
                # 反向传播与优化
                total_loss.backward()
                optimizer.step()

                if global_step % 10 == 0:
                    writer.add_scalars('Step/Total_Loss', {'Total': total_loss.item()}, global_step)
                    writer.add_scalars('Step/Individual_Loss', {
                        'Velocity': batch_v_loss.item() * 1000,
                        'Steer': batch_s_loss.item() * 1000
                    }, global_step)

                global_step += 1

                # 累加损失用于打印
                train_sum_loss += total_loss.item()
                train_sum_vel += batch_v_loss.item()
                train_sum_steer += batch_s_loss.item()
                # 打印当前 Batch 的损失
                if (batch_idx + 1) % 5 == 0:
                    print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], "
                          f"Total Loss: {total_loss.item():.6f}, V-Loss: {batch_v_loss.item()*1000:.10f}, S-Loss: {batch_s_loss.item()*1000:.10f}")
            # val片段
            model.eval()
            val_sum_loss, val_sum_vel, val_sum_steer = 0.0, 0.0, 0.0
            with torch.no_grad():
                for f_imgs, s_imgs, states, targets in val_loader:
                    f_imgs = [img.to(device) for img in f_imgs]
                    s_imgs = [img.to(device) for img in s_imgs]
                    states, targets = states.to(device), targets.to(device)
                    preds = model(f_imgs, s_imgs, states)

                    v_l = criterion_huber(preds[:, 0], targets[:, 0])
                    s_l = criterion_huber(preds[:, 1], targets[:, 1])
                    val_total = awl(v_l, s_l)

                    val_sum_loss += val_total.item()
                    val_sum_vel += v_l.item()
                    val_sum_steer += s_l.item()

            # 学习率调度
            scheduler.step()

            # 记录日志
            avg_loss = train_sum_loss / len(train_loader)
            avg_v = train_sum_vel / len(train_loader)
            avg_s = train_sum_steer / len(train_loader)
            avg_val_loss = val_sum_loss / len(val_loader)
            avg_val_vel = val_sum_vel / len(val_loader)
            avg_val_steer = val_sum_steer / len(val_loader)
            # 这里是输出平均的
            writer.add_scalar('Epoch/Train_Loss_Total', avg_loss, epoch + 1)
            writer.add_scalar('Epoch/Train_Loss_Velocity', avg_v, epoch + 1)
            writer.add_scalar('Epoch/Train_Loss_Steer', avg_s, epoch + 1)
            writer.add_scalar('Epoch/Val_Loss_Total', avg_val_loss, epoch + 1)
            writer.add_scalar('Epoch/Val_Loss_Velocity', avg_val_vel, epoch + 1)
            writer.add_scalar('Epoch/Val_Loss_Steer', avg_val_steer, epoch + 1)
            writer.add_scalars('Epoch/Adaptive_Weights', {
                'V_Weight': torch.exp(-awl.log_vars[0]).item(),
                'S_Weight': torch.exp(-awl.log_vars[1]).item()
            }, epoch + 1)



            with open(log_file, "a") as f:
                f.write(f"Epoch {epoch+1}: TrainTotal={avg_loss:.6f}, ValTotal={avg_val_loss:.6f}, "
                        f"ValVel={avg_val_vel:.6f}, ValSteer={avg_val_steer:.6f}\n")
            print(f"--- Epoch {epoch+1} 完成! 平均 Loss: {avg_loss:.6f} ---")
            # �� 修正4：定期保存模型
            if (epoch + 1) % 5 == 0:
                save_path = os.path.join(log_dir, f"DPAI_Driving_Epoch{epoch+1}.pth")
                torch.save({
                    'epoch': epoch+1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'awl_state_dict': awl.state_dict(),
                    'val_loss': avg_val_loss,
                }, save_path)
                print(f">>> 模型 checkpoint 已保存至 {save_path}")
        writer.close()
        print(">>> 训练全部完成！")
    else:
         print("加载数据集不训练")

## 测试代码，用于观察其中维度

In [ ]:

# 测试代码
if __name__ == '__main__':
    batch_size = 2
    embed_dim=384,    # 必须与 Swin 的输出通道一致
    num_patches=196,  # 对应 14x14 的 Patch 数量

    # 设定重要的自动驾驶类别权重
    my_weights = {0: 1.0, 1: 1.2, 7: 8.0, 11: 20.0}
    num_classes = 20

    # ==========================================
    # 1. 测试前端特征提取 (模拟真实图片输入)
    # ==========================================
    model = FrontViewSwinTiny(num_classes=num_classes, weights_dict=my_weights)

    # 生成假图像和掩码送入 Swin
    dummy_rgb = torch.randn(batch_size, 3, 224, 224)
    dummy_mask = torch.randint(0, num_classes, (batch_size, 1, 224, 224)).float()

    # 获取 Stage 4 的真实输出
    real_stage4_feat = model(dummy_rgb, dummy_mask)
    print(f"Swin Stage 4 真实输出形状: {real_stage4_feat.shape}")
    # 期望输出: [2, 49, 768]

    # ==========================================
    # 2. 测试时空交叉注意力融合
    # ==========================================
    fusion_module = TemporalCrossAttention(
        embed_dim=384,    # 必须与 Swin 的输出通道一致
        num_heads=12,     # 注意力头数，建议能被 embed_dim 整除
        num_patches=196,  # 对应 14x14 的 Patch 数量
        ffn_dim=1536      # 前馈网络的隐藏层维度
    )
    print(f"batch_size: {batch_size}, type: {type(batch_size)}")
    print(f"num_patches: {num_patches}, type: {type(num_patches)}")
    print(f"embed_dim: {embed_dim}, type: {type(embed_dim)}")
    # 修正方案：确保它们都是整数
    # 如果 num_patches 是 (14, 14)，你需要把它乘起来变成 196
    if isinstance(num_patches, tuple):
        num_patches = num_patches[0]
    # 确保 embed_dim 也是整数
    if isinstance(embed_dim, tuple):
        embed_dim = embed_dim[0]
    # 最终修正后的逻辑
    if isinstance(num_patches, tuple):
        # 如果元组里只有一个元素（如 (196,)），取索引 0
        # 如果有两个元素（如 (14, 14)），则相乘
        if len(num_patches) == 1:
            num_patches = num_patches[0]
        else:
            num_patches = num_patches[0] * num_patches[1]
    if isinstance(embed_dim, tuple):
        # 同样，对于 (384,) 取第一个元素
        embed_dim = embed_dim[0]
    # 现在定义随机张量就不会报错了

    # 模拟三帧 Stage 4 的输出 (当前帧 t, 过去帧 t1, t2)
    img_t = torch.randn(batch_size, num_patches, embed_dim)
    img_t1 = torch.randn(batch_size, num_patches, embed_dim)
    img_t2 = torch.randn(batch_size, num_patches, embed_dim)

    # 执行时空融合
    enhanced_feature = fusion_module(img_t, img_t1, img_t2)

    print(f"融合后的时空特征张量形状: {enhanced_feature.shape}")
    # 期望输出: [2, 49, 768]

## 开始训练代码

In [ ]:
# 找到你的原始数据相对于本 Notebook 的路径
# csv_files = [
#     'carla_data_collect/20260327_205825/csv/global_vehicle_data.csv'
#     # 如果有更多文件，可以继续添加这里
# ]
# # 启动训练
last_checkpoint = r"training_results1/DPAI_Driving_Epoch20.pth"
# train(input_files=csv_files, resume_path=last_checkpoint)

## 开始开环测试代码

In [ ]:
# SwinTiny.ipynb
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.cm as cm # 用于生成热力图配色
def visualize_attention_pil(model, dataset, idx, device, alpha=0.5):
    """
    使用 PIL 和 Matplotlib 渲染注意力热力图并进行叠加显示
    """
    model.eval()

    # 1. 获取并处理单条数据 (同前)
    front_imgs, side_imgs, state_seq, target = dataset[idx]

    input_f = [img.unsqueeze(0).to(device) for img in front_imgs]
    input_s = [img.unsqueeze(0).to(device) for img in side_imgs]
    input_state = state_seq.unsqueeze(0).to(device)

    # 2. 推理并提取注意力图 (t 对 t-1 帧的关注度)
    with torch.no_grad():
        # 注意：你需要先按照我上条建议修改了 main_model.py 以支持 return_attn=True
        preds, attn = model(input_f, input_s, input_state, return_attn=True)

    # 我们从 [1, 196, 392] 中取对前一帧 (t-1) 的关注度，聚合特征并转为 14x14
    # attn[0].mean(dim=0) 是查询当前帧所有 Patch 后对 K-V 的平均关注分布
    attn_weights = attn[0].mean(dim=0)[:196].reshape(14, 14).cpu().numpy()

    # 3. 归一化处理
    attn_normalized = (attn_weights - attn_weights.min()) / (attn_weights.max() - attn_weights.min() + 1e-8)

    # 4. 🌟 使用 PIL 和 Matplotlib 生成热力图
    # 将归一化数据转换为 PIL 灰度图并放大到 224x224
    attn_img = Image.fromarray(np.uint8(255 * attn_normalized), mode='L')
    attn_img = attn_img.resize((224, 224), resample=Image.BILINEAR)

    # 应用 Matplotlib 的 'jet' 配色方案 (给灰度图上色)
    # cm.jet 会返回一个形状为 (224, 224, 4) 的 RGBA 数组
    colormap = cm.get_cmap('jet')
    heatmap_rgba = colormap(np.array(attn_img) / 255.0)
    heatmap_rgb = np.delete(heatmap_rgba, 3, 2) # 去掉 Alpha 通道

    # 5. 🌟 读取原始图像并叠加 (PIL 混合模式)
    # raw_img 从 tensor 转为 0-255 的 PIL 对象
    raw_img_np = (front_imgs[2].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    # original_pil = Image.fromarray(raw_img_np)
    img_tensor = front_imgs[2].cpu()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    # 还原公式：原图 = (张量 * std) + mean
    img_denorm = img_tensor * std + mean
    img_denorm = torch.clamp(img_denorm, 0, 1) # 修正由于浮点计算误差导致的越界
    raw_img_np = (img_denorm.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    original_pil = Image.fromarray(raw_img_np)
    heatmap_pil = Image.fromarray(np.uint8(255 * heatmap_rgb))

    # 使用 PIL 的 blend 进行半透明融合: (1 - alpha) * original + alpha * heatmap
    blended_img = Image.blend(original_pil, heatmap_pil, alpha=alpha)

    # 6. 显示结果
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(original_pil); plt.title("Original Frame (t-1)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(blended_img); plt.title("Attention Heatmap (PIL)")
    plt.axis('off')
    plt.savefig(f'{idx}')

    plt.show()

def evaluate_and_save_to_csv(checkpoint_path, input_files, result_csv_name='inference_results.csv', evaling = True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    output_csv = 'processed_data.csv'
    scaler_json_path = 'scaler_params.json'
    scaler_params = prepare_multiple_datasets_and_scaler(
        input_files=input_files,
        output_csv=output_csv,
        scaler_json_path=scaler_json_path,
        seq_length=9)
    print('测试结果输出成果')
    
    # 1. 加载归一化参数 (用于还原真实数值)
    with open('scaler_params.json', 'r') as f:
        scaler_params = json.load(f)
    
    # 2. 初始化模型并加载权重
    print(f">>> 正在加载模型权重: {checkpoint_path}")
    unet_pth = r"E:\Laboratory files\code_project\D2D\SwinTiny\dpai_unet.pth"
    model = EndToEndDrivingPipeline(unet_weight_path=unet_pth, device=device, output_dim=2).to(device)
    
    # 支持加载 dict 格式或纯权重格式的 checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    model.eval()

    # 3. 准备测试数据 (通常使用全部数据或验证集数据)
    data_root = r'E:\Laboratory files\code_project\D2D' 
    # 先生成一个临时推理配置
    dataset = ProcessedDrivingDataset(csv_file='processed_data.csv', root_dir=data_root)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False) # Inference 时设为 1 方便观察
    if evaling:
        results = []
        print(">>> 开始执行推理...")
        with torch.no_grad():
            for i, (front_imgs, side_imgs, state_seq, target) in enumerate(dataloader):
                # 数据推送到设备
                front_imgs = [img.to(device) for img in front_imgs]
                side_imgs = [img.to(device) for img in side_imgs]
                state_seq = state_seq.to(device)

                # 模型预测
                preds = model(front_imgs, side_imgs, state_seq) # [1, 2]

                # 将 Tensor 转回 Numpy
                pred_val = preds.squeeze(0).cpu().numpy()
                gt_val = target.squeeze(0).cpu().numpy()

                # 4. 🌟 关键：反向归一化 (还原真实物理量)
                # 根据你 dataset_create.py 里的顺序：[0]是速度，[1]是转角
                v_p, s_p = inverse_transform(pred_val, ['velocity', 'steer'], scaler_params)
                v_gt, s_gt = inverse_transform(gt_val, ['velocity', 'steer'], scaler_params)

                results.append({
                    'Index': i,
                    'GT_Velocity': v_gt,
                    'GT_Steer': s_gt,
                    'Pred_Velocity': v_p,
                    'Pred_Steer': s_p,
                    'Error_V': abs(v_gt - v_p),
                    'Error_S': abs(s_gt - s_p)
                })

                if (i + 1) % 50 == 0:
                    print(f"已完成 {i+1} 条数据的对比。")

        # 5. 保存结果到 CSV
        res_df = pd.DataFrame(results)
        res_df.to_csv(result_csv_name, index=False)
        print(f"\n✅ 推理对比完成！结果已保存至: {result_csv_name}")
    else:
        visualize_attention_pil(model, dataset, idx=150, device=device)
        print("只加载了eval数据集")

# --- 启动测试 ---
# 请修改为你想测试的权重路径
# my_checkpoint = r"E:\Laboratory files\code_project\D2D\SwinTiny\SwinTiny_logs\DPAI_Driving_Epoch30.pth"
my_checkpoint = r"E:\Laboratory files\code_project\D2D\SwinTiny\SwinTiny_logs\Swin_Epoch20.pth"
csv_files = [r"E:\Laboratory files\code_project\D2D\carla_data_collect\20260403_213808\csv\global_vehicle_data.csv"]
evaluate_and_save_to_csv(my_checkpoint, input_files=csv_files, evaling=False)


## 开始单张图片可视化